# 05 — SHAP Analysis (Module 4)

Global and local explainability of the champion LightGBM demand forecasting model.

- Dataset: `data/processed/walmart_master.parquet` (Module 3 output)
- Model: `reports/model/champion.joblib` (Module 4 training, champion by WMAE)
- Outputs: `reports/xai/shap_*.png` and `reports/xai/shap_feature_importance.csv`

In [ ]:
import sys
sys.path.insert(0, '..')

import joblib
import pandas as pd
import shap
shap.initjs()

In [ ]:
from src.models.train import FEATURES, PROCESSED, SPLIT_DATE

model = joblib.load('../reports/model/champion.joblib')
df = pd.read_parquet('../' + str(PROCESSED).split('walmart-demand-forecasting/')[-1])
df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
for col in ['sales_lag_1w', 'sales_lag_4w', 'sales_lag_52w']:
    df[col] = df.groupby(['Store', 'Dept'])[col].ffill().fillna(df[col].median())
df['IsHoliday'] = df['IsHoliday'].astype(int)
df = df.dropna(subset=FEATURES).reset_index(drop=True)
test = df[df['Date'] >= SPLIT_DATE]
X = test[FEATURES].sample(5000, random_state=42)
X.head()

## Global importance

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)
shap.summary_plot(shap_values, X, feature_names=FEATURES, plot_type='bar')

In [ ]:
shap.summary_plot(shap_values, X, feature_names=FEATURES)

## Local explanation (waterfall)

In [ ]:
row = X.iloc[[0]]
sv_row = explainer(row)
shap.plots.waterfall(sv_row[0])

## Business reading

- `sales_lag_52w` dominates: last-year-same-week is the strongest single signal, consistent with Fildes et al. (2022).
- `Size` and `Dept` are structural fixed effects that calibrate the baseline for each store-department.
- `IsHoliday` and `MarkDown*` move individual predictions by a smaller but non-trivial amount.
- No feature shows a counter-intuitive sign (e.g. higher CPI pushing sales up without context), so the model's learned associations are business-interpretable.